# 00. 任务分解与依赖

将复杂目标拆解为可独立验证的子任务，是 Agent 规划的第一步。

**核心原则**：
- 每个子任务必须有明确的完成条件（acceptance criteria）
- 依赖关系必须显式声明
- 优先识别可并行的任务

> **检查点**：能解释为什么「没有完成条件的分解」比不分解更危险。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
from dataclasses import dataclass, field
from collections import deque

@dataclass
class Task:
    id: str; desc: str; deps: list = field(default_factory=list)

def topo_rounds(tasks: list[Task]) -> list[list[Task]]:
    """拓扑排序：将任务分组为可并行执行的 rounds"""
    name_map = {t.id: t for t in tasks}
    indeg = {t.id: len(t.deps) for t in tasks}
    children = {t.id: [] for t in tasks}
    for t in tasks:
        for d in t.deps:
            children[d].append(t.id)
    
    q = deque(tid for tid, d in indeg.items() if d == 0)
    rounds = []
    while q:
        round_tasks = [name_map[q.popleft()] for _ in range(len(q))]
        rounds.append(round_tasks)
        for t in round_tasks:
            for child in children[t.id]:
                indeg[child] -= 1
                if indeg[child] == 0:
                    q.append(child)
    return rounds

tasks = [Task("T1","Design schema"), Task("T2","Setup project",["T1"]),
         Task("T3","Implement models",["T1"]), Task("T4","Implement API",["T2","T3"]),
         Task("T5","Write tests",["T4"])]

rounds = topo_rounds(tasks)
print(f"Task count: {len(tasks)}, Rounds: {len(rounds)}")
for i, rnd in enumerate(rounds, 1):
    names = ", ".join(t.id for t in rnd)
    deps = set(d for t in rnd for d in t.deps)
    print(f"  Round {i}: [{names}] (inputs: {deps or 'none'})")

Task count: 5, Rounds: 4
  Round 1: [T1] (inputs: none)
  Round 2: [T2, T3] (inputs: {'T1'})
  Round 3: [T4] (inputs: {'T2', 'T3'})
  Round 4: [T5] (inputs: {'T4'})


## 依赖图 vs 线性列表

为什么需要 DAG 而非简单列表？

| 方式 | 优点 | 缺点 |
|------|------|------|
| 线性列表 | 简单、易调试 | 无法利用并行，修改一个步骤影响全局 |
| DAG | 天然并行、局部修改不影响整体 | 需要依赖管理、可能导致死锁 |

> **检查点**：能画出给定目标的依赖关系 DAG。

## 真实 API 实验

让真实 LLM 分解一个目标并标注依赖，再用拓扑排序排出执行顺序。


In [3]:
# 真实任务分解（结构化输出）+ 拓扑排序
data = chat_json(client, """把目标「给博客添加评论功能」分解为 4-6 个子任务。
返回 JSON: {"tasks": [{"id": "t1", "title": "...", "depends_on": []}, ...]}""")
tasks = data.get("tasks", [])
for t in tasks:
    print(f"{t['id']}: {t['title']}  (依赖: {t.get('depends_on', [])})")

# 拓扑排序
done, order = set(), []
while len(order) < len(tasks):
    for t in tasks:
        if t["id"] not in done and all(d in done for d in t.get("depends_on", [])):
            order.append(t["id"]); done.add(t["id"])
    else:
        continue
    break
print("\n执行顺序:", " → ".join(order))


t1: 设计评论数据模型与数据库表  (依赖: [])
t2: 实现评论的CRUD REST API  (依赖: ['t1'])
t3: 开发评论前端展示与表单组件  (依赖: ['t2'])
t4: 集成用户认证与权限控制（仅登录用户可评论）  (依赖: ['t2'])
t5: 实现评论管理后台（审核、删除、编辑）  (依赖: ['t2', 't4'])
t6: 编写集成测试并部署上线  (依赖: ['t3', 't5'])

执行顺序: t1 → t2 → t3 → t4 → t5 → t6


## 练习

1. 设计一个「写博客文章」的任务分解，标注所有依赖关系。哪几步可以并行？
2. 如果 T3（实现模型）失败，T4 和 T5 会怎样？设计 re-plan 策略。
3. 实现一个 `validate(task)` 函数，检查一个 task 的完成条件是否满足。

> **检查点**：能独立为一个真实项目（如「上线一个新功能」）完成 DAG 分解。